<a href="https://colab.research.google.com/github/sanyamsh7/Agentic-RAG-Pipeline-From-Scratch/blob/main/Module6_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODULE 6: THE AGENTIC LAYER - Adding Intelligence to RAG

## Core Concepts:
```
Regular RAG:
  Every query → Search documents → Generate answer

Agentic RAG:
  Every query → AGENT DECIDES → Search OR Direct answer
                               ↓
                    "Do I actually need documents?"
```
You'll learn:
- What makes RAG "agentic"
- Routing logic (when to search vs answer directly)
- Building an agent controller
- Complete agentic RAG pipeline
- Real-world examples


In [ ]:
# installing packages
!pip install -q langchain langchain-community langchain-chroma
!pip install -q transformers torch sentence-transformers chromadb

In [ ]:
# importing libraries
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.language_models.llms import LLM
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from typing import Any, List, Optional, Dict, Literal
from pydantic import Field
import re
import warnings
warnings.filterwarnings('ignore')


## Part-1: Understanding Agentic RAG

🤔 THE PROBLEM WITH TRADITIONAL RAG

Traditional RAG (Non-Agentic):

User: "Hello!"

System: [Searches documents] → "Hello! According to document X..."

→ WASTE OF TIME! No need to search for a greeting!

User: "What's 2 + 2?"

System: [Searches documents] → "I don't have information about that"

→ SILLY! The LLM knows basic math!

User: "Summarize my document about AI"

System: [Searches documents] → Perfect! This SHOULD search!

THE INSIGHT:

NOT EVERY QUERY NEEDS DOCUMENT RETRIEVAL!

Some queries can be answered:

✅ From the LLM's general knowledge (greetings, math, common facts)

✅ Directly without searching (conversations, clarifications)

Some queries MUST use documents:

✅ Questions about YOUR specific documents

✅ Domain-specific information

✅ Recent/proprietary knowledge

AGENTIC RAG = SMART ROUTING


                    User Query
                        ↓
                [AGENT DECISION]
                        ↓
            ┌───────────┴───────────┐
            ↓                       ↓
    [NEEDS DOCUMENTS?]         [GENERAL QUERY?]
            ↓                       ↓
    Retrieve → Generate      Direct Answer
            ↓                       ↓
        Answer                  Answer

The agent DECIDES the best path!

BENEFITS:

✅ Faster responses (no unnecessary searches)

✅ Lower costs (fewer embedding computations)

✅ Better UX (natural conversation flow)

✅ Smarter system (context-aware routing)

## Part-2: Routing Strategies

🧠 THREE ROUTING APPROACHES

1. KEYWORD-BASED ROUTING (Simple & Fast)

   Check for keywords in the query
   
   Document keywords: ["document", "pdf", "file", "summarize", "find"]

   General keywords: ["hello", "hi", "thanks", "what is", "how to"]
   
   Pros: Fast, no LLM call needed

   Cons: Limited, can miss context
   
   Example:

   "Tell me about the PDF" → HAS "PDF" → Search documents ✅

   "Hello!" → HAS "hello" → Direct answer ✅

   "What's in section 3?" → HAS "section" → Search documents ✅

2. LLM-BASED ROUTING (Smart & Accurate)

   Ask a small LLM to classify the query
   
   Prompt: "Does this query need document retrieval? Yes/No: [query]"
   
   Pros: Understands context and intent

   Cons: Requires extra LLM call (slower)
   
   Example:

   LLM sees: "What does the report say about Q3 sales?"

   LLM decides: "Yes - needs documents" → Search ✅

3. HYBRID ROUTING (Best of Both)

   Use keywords first, LLM as backup
   
   1. Check keywords (fast)

   2. If uncertain → Ask LLM (accurate)
   
   Pros: Fast + Accurate
   
   Cons: More complex logic
   
   This is what production systems use!

FOR THIS MODULE: We'll implement all three! 🎯

### Load LLM


In [ ]:
class FlanT5LLM(LLM):
  model_name: str = Field(default = "google/flan-t5-large")
  model: Any = Field(default= None, exlude = True)
  tokenizer: Any = Field(default= None, exlude = True)
  max_length: int = Field(default = 200)
  temperature: float = Field(default = 0.1)
  repetition_penalty: float = Field(default = 1.2)
  device: str = Field(default = "cpu")

  class Config:
    arbitrary_types_allowed = True

  def __init__(self, **kwargs):
    super().__init__(**kwargs)
    print("Loading -- ", self.model_name)
    self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
    # Check for GPU and move model to it
    if torch.cuda.is_available():
      self.device = "cuda"
      print(f"Moving model to GPU: {self.device}")
    else:
      self.device = "cpu"
      print("Running on CPU.")
    self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name).to(self.device)
    print("Model Loaded .. \n")

  @property
  def _llm_type(self) -> str:
    return "flan-t5"

  def _call(
      self,
      prompt: str,
      stop: Optional[List[str]] = None,
      run_manager: Optional[Any] = None,
      **kwargs: Any
  ) -> str:
      inputs = self.tokenizer(prompt, return_tensors = "pt", max_length = 512, truncation = True).to(self.device) # Move inputs to device
      with torch.no_grad():
        outputs = self.model.generate(
            inputs.input_ids,
            max_length = self.max_length,
            temperature = self.temperature,
            do_sample = True if self.temperature> 0 else False,
            repetition_penalty = self.repetition_penalty,
            early_stopping = True
        )
      return self.tokenizer.decode(outputs[0], skip_special_tokens = True).strip()
  @property
  def _identifying_params(self) -> Dict[str, Any]:
    return {
        "model_name": self.model_name,
        "device": self.device
    }

In [1]:
# initalize LLM
llm = FlanT5LLM(temperature = 0)

### Create Knowledge Base

In [ ]:
documents = [
    Document(
        page_content="""Machine learning is a subset of artificial intelligence
        that enables systems to learn from data. Our company uses machine learning
        for customer segmentation and predictive analytics. Key applications include
        churn prediction and recommendation systems.""",
        metadata={"source": "ml_report.pdf", "page": 1, "type": "technical"}
    ),
    Document(
        page_content="""Q3 2024 Sales Report: Total revenue reached $5.2M, up 15%
        from Q2. Top performing products were Enterprise AI Suite ($2.1M) and
        Data Analytics Platform ($1.8M). Customer acquisition cost decreased by 12%.""",
        metadata={"source": "q3_sales.pdf", "page": 1, "type": "business"}
    ),
    Document(
        page_content="""Company Policy: Remote work is allowed up to 3 days per week.
        Employees must submit weekly status reports by Friday 5 PM. Annual performance
        reviews occur in December. Health insurance coverage begins after 90 days.""",
        metadata={"source": "hr_policies.pdf", "page": 3, "type": "policy"}
    ),
]

In [2]:
# setup vector store
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embeddings,
    collection_name = "agentic_rag"
)
retriever = vectorstore.as_retriever(search_kwargs={"k":2})

## Part-3: Strategy 1 - Keyword based routing



In [ ]:
class KeywordRouter:
  """
  simple keyword-based routing, fast but limited - good for clear-cut cases"""
  def __init__(self):
    # keywords that indicate document search is needed
    self.search_keywords = [
        "document", "pdf", "file", "report", "policy",
            "summarize", "summary", "find", "search", "show me",
            "what does", "according to", "in the", "from the",
            "q3", "sales", "revenue", "our company"
    ]
    # keywords that indicate direct answer is fine
    self.direct_keywords = [
            "hello", "hi", "hey", "thanks", "thank you",
            "what is", "who is", "define", "explain",
            "how to", "why", "when", "where"
        ]

  def route(self, query:str) -> Literal["search", "direct"]:
    """
    deciding routing based on keywords
    returns: "search" or "direct"
    """
    query_lower = query.lower()
    # checking for search keywords - brute approach
    for keyword in self.search_keywords:
      if keyword in query_lower:
        return "search"

    # check for direct keywords ( only if no search keywords found )
    for keyword in self.direct_keywords:
      if keyword in query_lower:
        return "direct"

    # default: search(safer to retrieve than miss information )
    return "search"


In [ ]:
# testing keyword routing
router = KeywordRouter()
test_queries = [
    "Hello, how are you?",
    "What were our Q3 sales?",
    "What is machine learning?",
    "Summarize the sales report",
    "Thanks for your help!",
]

for query in test_queries:
  decision = router.route(query)
  icon = "📄" if decision == "search" else "💬"
  print(f"{icon} '{query}'")
  print(f"→ Decision: {decision.upper()}")
  print()

💬 'Hello, how are you?'
→ Decision: DIRECT

📄 'What were our Q3 sales?'
→ Decision: SEARCH

💬 'What is machine learning?'
→ Decision: DIRECT

📄 'Summarize the sales report'
→ Decision: SEARCH

💬 'Thanks for your help!'
→ Decision: DIRECT



LIMITATIONS:

⚠️  "Tell me about our ML approach" → Might miss "our" context

⚠️  "How does Einstein's theory work?" → Might incorrectly search

## Part-4: Strategy 2 - LLM based routing

In [ ]:
class LLMRouter:
  """
  LLM-based routing, more accurate but slower- understands context
  """
  def __init__(self, llm: LLM):
    self.llm = llm

    # classification prompt
    self.routing_prompt = """You are a routing assistant. Decide if the query needs document retrieval

    Answer with ONLY "SEARCH" or "DIRECT":

SEARCH if:
- Query asks about specific documents/files/reports
- Query needs company-specific information
- Query asks about proprietary/recent data
- Query mentions words like our, the document, the report

DIRECT if:
- General greetings (hello, hi, thanks)
- General knowledge questions (what is X, who is Y)
- Math/calculations
- Conversational responses

Query: {query}

Answer (SEARCH or DIRECT):"""

  def route(self, query:str) -> Literal["search", "direct"]:
    """
    deciding routing based on LLM
    returns: "search" or "direct"
    """
    prompt = self.routing_prompt.format(query = query)
    response = self.llm.invoke(prompt).strip().upper()
    print(response)
    # parse response
    if "SEARCH" in response:
      return "search"
    elif "DIRECT" in response:
      return "direct"
    else:
      return "search"

In [ ]:
# testing llm router
llm_router  = LLMRouter(llm)

In [ ]:
test_queries = [
    "Hello!",
    "What were our Q3 sales numbers?",
    "What is the capital of France?",
    "Tell me about our company's machine learning approach", # had to add the word - our company's for better context else it comes in general knowledge
    "How much is 25 * 4?",
]

for query in test_queries:
    decision = llm_router.route(query)
    icon = "📄" if decision == "search" else "💬"
    print(f"{icon} '{query}'")
    print(f"→ Decision: {decision.upper()}")
    print()

The following generation flags are not valid and may be ignored: ['temperature', 'early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


DIRECT
💬 'Hello!'
→ Decision: DIRECT

SEARCH
📄 'What were our Q3 sales numbers?'
→ Decision: SEARCH

DIRECT
💬 'What is the capital of France?'
→ Decision: DIRECT

SEARCH
📄 'Tell me about our company's machine learning approach'
→ Decision: SEARCH

DIRECT
💬 'How much is 25 * 4?'
→ Decision: DIRECT



OBSERVATIONS:

✅ Better context understanding

✅ Catches "our" indicating company-specific info

✅ Distinguishes general knowledge vs. document queries

✅ More nuanced decisions

TRADE-OFF:

⚠️  Slower (extra LLM call)

⚠️  Costs more (if using paid APIs)

✅ But much more accurate!

## Part-5: Strategy 3 - Hybrid approach

In [ ]:
class HybridRouter:
  """
  combines keyword and LLM routing
  fast when obvious, smart when uncertain
  """

  def __init__(self, llm: LLM):
    self.keyword_router = KeywordRouter()
    self.llm_router = LLMRouter(llm)

    # queries that are obviously one or other
    self.obvious_direct = [
        r"^(hello|hi|hey|thanks|thank you)",
        r"^(goodbye|bye|see you)",
        r"^\d+\s*[\+\-\*/]\s*\d+",  # Math expressions
    ]

    self.obvious_search = [
            r"(summarize|summary of).*(document|pdf|file|report)",
            r"(what does|what is in).*(document|report|file)",
            r"(our|the).*(sales|revenue|policy|report)",
        ]

  def route(self, query:str) -> Literal["search", "direct"]:
    """
    hybrid routing decisions
    1. check for obvious cases(fast)
    2. if uncertain, ask LLM (accurate)
    """
    query_lower = query.lower()
    for pattern in self.obvious_direct:
      if re.search(pattern, query_lower):
        return "direct"

    for pattern in self.obvious_search:
      if re.search(pattern, query_lower):
        return "search"

    # uncertain ask llm
    return self.llm_router.route(query)

In [ ]:
# testing hybrid routing
hybrid_router = HybridRouter(llm)

test_queries = [
    "Hi there!",  # Obvious direct
    "Summarize the sales report",  # Obvious search
    "Tell me about machine learning in our company",  # Uncertain → LLM
    "What is 15 + 27?",  # Obvious direct
    "What's the remote work policy?",  # Uncertain → LLM
]

for query in test_queries:
    decision = hybrid_router.route(query)
    icon = "📄" if decision == "search" else "💬"
    print(f"{icon} '{query}'")
    print(f"→ Decision: {decision.upper()}")
    print()


💬 'Hi there!'
→ Decision: DIRECT

📄 'Summarize the sales report'
→ Decision: SEARCH

SEARCH
📄 'Tell me about machine learning in our company'
→ Decision: SEARCH

DIRECT
💬 'What is 15 + 27?'
→ Decision: DIRECT

📄 'What's the remote work policy?'
→ Decision: SEARCH



HYBRID BENEFITS:

✅ Fast for obvious cases (no LLM call)

✅ Smart for uncertain cases (LLM decides)

✅ Best accuracy-speed trade-off

✅ Production-ready approach

## Part-6: Building the Agentic RAG Pipeline

In [ ]:
class AgenticRAG:
  """
  complete agentic rag system with intelligen routing

  features:
  - smart routing ( hybrid approach )
  - document retrieval when needed
  - direct answers when appropriate
  - source citations
  - conversation tracking
  """
  def __init__(
      self,
      documents: List[Document],
      llm: LLM,
      routing_strategy: Literal["keyword", "llm", "hybrid"] = "hybrid"
  ):
      print("Initializing Agentic RAG ...")
      self.llm = llm
      # setting up vector store
      embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
      self.vectorstore = Chroma.from_documents(
          documents = documents,
          embedding = embeddings,
          collection_name = "agentic_rag_final"
      )
      self.retriever = self.vectorstore.as_retriever(search_kwargs = {"k":2})
      # setup router
      if routing_strategy == "keyword":
        self.router = KeywordRouter()
      elif routing_strategy == "llm":
        self.router = LLMRouter(llm)
      else:
        self.router = HybridRouter(llm)
      # RAG prompt
        self.rag_template = """Use ONLY the context to answer. If not in context, say so.

                            Context:
                            {context}

                            Question: {question}

                            Answer:"""
        self.rag_prompt = PromptTemplate.from_template(self.rag_template)
        # direct prompt
        self.direct_template = """Answer the question concisely:

                        Question: {question}

                        Answer:"""
        self.direct_prompt = PromptTemplate.from_template(self.direct_template)

        print("Agentic RAG ready!")
  def _format_docs(self, docs: List[Document]) -> str:
    formatted = []
    for i, doc in enumerate(docs, 1):
      source = doc.metadata.get("source", "Unkown")
      formatted.append(f"[Source {i}: {source}]\n{doc.page_content}")
    return "\n\n".join(formatted)

  def ask(self, question:str, verbose: bool = True) -> Dict[str, Any]:
    """
    answer question with intelligent routing
    returns: dict with answer and metadata
    """
    # step 1 - route the query
    decision = self.router.route(question)
    if verbose:
      print("Routing decision:", decision.upper())
    # step 2 - execute based on decision
    if decision == "search":
      # retrieve documents
      docs = self.retriever.invoke(question)
      context = self._format_docs(docs)
      #build prompt
      prompt = self.rag_prompt.format(context=context, question=question)
      # generate
      answer = self.llm.invoke(prompt)

      return {
          "question": question,
                "answer": answer,
                "route": "search",
                "sources": [
                    {
                        "source": doc.metadata.get("source", "Unknown"),
                        "type": doc.metadata.get("type", "Unknown")
                    }
                    for doc in docs
                ]
      }
    else:  # direct
            # Direct answer
            prompt = self.direct_prompt.format(question=question)
            answer = self.llm.invoke(prompt)

            return {
                "question": question,
                "answer": answer,
                "route": "direct",
                "sources": []
            }




In [3]:
# testing the agentic rag system
agentic_rag = AgenticRAG(
    documents=documents,
    llm=llm,
    routing_strategy="hybrid"  # Using best strategy
)
test_queries = [
    # Greetings (should go direct)
    "Hello!",

    # Document-specific (should search)
    "What were our Q3 sales numbers?",

    # General knowledge (should go direct)
    "What is machine learning?",

    # Company-specific (should search)
    "What's our remote work policy?",

    # Math (should go direct)
    "What's 144 divided by 12?",

    # Document query (should search)
    "Tell me about our AI applications",
]

for query in test_queries:
    print(f"\n{'='*70}")
    print(f"Query: {query}")
    print("-" * 70)

    result = agentic_rag.ask(query, verbose=True)

    print(f"📝Answer: {result['answer']}")

    if result['sources']:
        print(f"📚Sources: {', '.join([s['source'] for s in result['sources']])}")
    else:
        print(f"📚Sources: None (direct answer)")

# PART 8: COMPARING AGENTIC VS NON-AGENTIC

SIDE-BY-SIDE COMPARISON

Query: "Hello, how are you?"
----------------------------
NON-AGENTIC RAG:
  1. Embed query → vector search
  2. Retrieve documents (waste!)
  3. Build RAG prompt with irrelevant context
  4. Generate answer
  ⏱️  Time: ~500ms
  💰 Cost: Full embedding + retrieval + generation

AGENTIC RAG:
  1. Router: "This is a greeting" → DIRECT
  2. Generate answer directly
  ⏱️  Time: ~100ms (5x faster!)
  💰 Cost: Just generation (80% savings!)

Query: "What were Q3 sales?"
----------------------------
NON-AGENTIC RAG:
  1. Always searches (correct in this case)
  2. Retrieves documents
  3. Generates answer
  ⏱️  Time: ~500ms

AGENTIC RAG:
  1. Router: "Document query" → SEARCH
  2. Retrieves documents
  3. Generates answer
  ⏱️  Time: ~500ms (same)
  
EFFICIENCY GAINS:
-----------------
Assume 100 queries:
- 40% are direct (greetings, general knowledge, math)
- 60% need documents

NON-AGENTIC: 100 searches = 100 × cost

AGENTIC: 60 searches = 60 × cost = 40% SAVINGS! 💰

PLUS:

✅ Faster responses for direct queries

✅ Better user experience

✅ More natural conversation

✅ Reduced server load

THIS IS WHY AGENTIC RAG IS THE FUTURE! 🚀

# PART 9: PRODUCTION BEST PRACTICES

🏭 DEPLOYING AGENTIC RAG IN PRODUCTION

1. ROUTING STRATEGY SELECTION
   ---------------------------
   Low-traffic app (<1000 queries/day):
   → Use LLM routing (accuracy over speed)
   
   High-traffic app (>10,000 queries/day):
   → Use hybrid routing (balance)
   
   Cost-sensitive:
   → Use keyword routing (cheapest)

2. MONITORING & ANALYTICS
   -----------------------
   Track these metrics:
   
   - Route distribution (% direct vs search)
   - Route accuracy (manual review sample)
   - Response times by route
   - Cost per query
   - User satisfaction by route
   
   Example dashboard:
   ```
   Today's Stats:
   - Total queries: 1,247
   - Direct route: 42% (524 queries)
   - Search route: 58% (723 queries)
   - Avg response time: 285ms
   - Cost savings: 42% vs non-agentic
   ```

3. CONTINUOUS IMPROVEMENT
   -----------------------
   - Review misrouted queries weekly
   - Update keyword lists
   - Retrain routing LLM on edge cases
   - A/B test routing strategies
   - Collect user feedback

4. FALLBACK STRATEGIES
   --------------------
   What if routing fails?
   
   → Default to SEARCH (safer)
   → Log the failure for review
   → Alert if failure rate > 5%

5. CACHING
   --------
   Cache common queries:
   
   "Hello" → Always direct
   "Thanks" → Always direct
   "What's our revenue?" → Always search
   
   → Skip router entirely for cached queries!

6. MULTI-LANGUAGE SUPPORT
   -----------------------
   Router needs to understand all languages:
   
   Option 1: Translate query to English first
   Option 2: Multi-lingual router
   Option 3: Language-specific keyword lists

7. SECURITY & PRIVACY
   -------------------
   - Sanitize queries before routing
   - Don't log sensitive information
   - Ensure router can't be tricked
   - Rate limit routing calls

# FINAL: KEY TAKEAWAYS

1. WHAT IS AGENTIC RAG?

   ✅ Intelligent routing before retrieval

   ✅ Decides: Search documents OR answer directly

   ✅ Not every query needs document retrieval!

2. THREE ROUTING STRATEGIES:
   
   KEYWORD-BASED:

   ✅ Fastest (no LLM call)

   ⚠️  Limited accuracy
   → Use for: Simple cases, keyword-heavy domains
   
   LLM-BASED:

   ✅ Most accurate (understands context)

   ⚠️  Slower (extra LLM call)
   → Use for: Complex queries, nuanced decisions
   
   HYBRID:

   ✅ Best balance (fast + accurate)

   ✅ Production-ready
   → Use for: Most applications ⭐

3. THE COMPLETE FLOW:
  ```
   User Query
       ↓
   [AGENT ROUTER]
       ↓
   ┌──────┴──────┐
   ↓             ↓
   SEARCH      DIRECT
   ↓             ↓
   Embed       Generate
   ↓
   Vector DB
   ↓
   Retrieve
   ↓
   Generate
   ↓
   Answer ←──────┘
```
4. BENEFITS:

   ✅ 40-50% cost savings (fewer retrievals)

   ✅ 2-5x faster for direct queries

   ✅ Better user experience (natural flow)

   ✅ More scalable (reduced load)

5. PRODUCTION CHECKLIST:

   □ Choose routing strategy

   □ Set up monitoring

   □ Implement caching

   □ Add fallback logic

   □ Track metrics

   □ Continuous improvement

You now understand:

✅ How to load and chunk documents

✅ How embeddings capture meaning

✅ How vector databases enable fast search

✅ How LLMs generate answers

✅ How routing adds intelligence

✅ How to deploy in production